In [1]:
import sys
sys.path.insert(0, '../gofher')

import os
import matplotlib.image as mpimg
import numpy as np

from gofher import run_gofher,run_gofher_with_parameters
from visualize import visualize
from file_helper import write_csv,check_if_folder_exists_and_create
from spin_parity import read_spin_parity_galaxies_label_from_csv, standardize_galaxy_name
from sparcfire import read_sparcfire_galaxy_csv, get_ref_band_and_gofher_params

In [2]:
survery_to_use = "sdss" #Note: for sdss we are not using u do to poor quality

BANDS_IN_ORDER = ['g','r','i','z'] #Important: Must stay in order of BLUEST to REDDEST Waveband (Editting this will cause gofher to no longer correctly evaluate redder side of galaxy)
REF_BANDS_IN_ORDER = ['r','i','z','g'] #The prefernce each waveband being choosen as refernce band from highest priority to lowest priority

In [3]:
#figure_to_run_on = "figure8"
#figure_to_run_on = ["figure8","figure9"] #finish "figure11", "figure10",
#figure_to_run_on = ["figure8","figure9","figure10","figure11"]
types_of_runs = ["fixed-center","fixed"]

#figure_to_run_on = ["figure10","figure8"] 
#figure_to_run_on = ["figure9"]
##types_of_runs = ["inital"]

steps = 4
#bulge_disk_fs = np.linspace(0.0,1.0,steps+1)
bulge_disk_fs = [0.75]
#bulge_disk_fs = [0.5]
figure_to_run_on = ["figure8","figure10","figure11","figure9"]


#panstarrs:
# = None #None or a positive integer

#sdss:
#NOTE: sdss needs to flip color image
bin_size = 4 #should be 4
#Source: "The median seeing of all SDSS imaging data (using the psfWidth metric) is 1.32 arcseconds in the r-band."
#"The pixel size in the Sloan Digital Sky Survey (SDSS) is 0.396 arcseconds per pixel" - https://classic.sdss.org/dr3/instruments/imager/
bin_prior_to_param_fitting = True

In [4]:
generate_verbose_csv = True #True
generate_ebm_csv = True
generate_params_csv = True #True
generate_visualization = True
save_visualization = True

In [5]:
#Important: Make sure you update these values:
path_to_catalog_data = "..\\..\\spin-parity-catalog-data"
#path_to_output = "..\\..\\gofher-data\\panstarrs\\default_ellipse_mask_fitting" #- https://www.sdss4.org/dr17/imaging/other_info/
#path_to_output = "E:\\grad_school\\research\\spring_2025\\sdss\\default_ellipse_mask_fitting"

def get_path_to_output(figure_to_run_on,run_type="",bulge_disk_f=1.0):
    #TODO check type of 
    path_to_output_base = os.path.join("E:\\grad_school\\research\\spring_2025",survery_to_use,run_type)
    folder = run_type

    if run_type != "":
        folder += "_{}".format(str(bulge_disk_f).replace(".","_"))
    
    return os.path.join(path_to_output_base,folder)

def get_visulization_save_path_folder(figure_to_run_on,run_type="",bulge_disk_f=1.0):
    path_to_output_base = os.path.join("E:\\grad_school\\research\\spring_2025",survery_to_use,run_type)
    folder = run_type

    if run_type != "":
        folder += "_{}".format(str(bulge_disk_f).replace(".","_"))
    
    return os.path.join(path_to_output_base,folder,figure_to_run_on)

In [6]:
def get_fits_path(name,band,figure_to_run_on):
    """the file path of where existing fits files can be found"""
    return os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on,name,"{}_{}.fits".format(name,band))

def get_color_image_path(name,figure_to_run_on):
    file_type = "png"
    if survery_to_use == "panstarrs": file_type = "jfif"
    return os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on,name,"{}_color.{}".format(name,file_type))

def get_path_to_catalog_csv(figure_to_run_on):
    return os.path.join(path_to_catalog_data,"catalog","{}.csv".format(figure_to_run_on))

In [7]:
def get_paper_dark_side_labels(figure_to_run_on):
    return read_spin_parity_galaxies_label_from_csv(get_path_to_catalog_csv(figure_to_run_on))

def get_galaxies(figure_to_run_on):
    return os.listdir(os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on))

def get_sparcfire_path(figure_to_run_on): #temp
    table_dict = {"figure8":"2","figure9":"3","figure10":"4","figure11":"5"}
    pa_key = table_dict.get(figure_to_run_on,"")
    if pa_key == "": raise ValueError("Invlid figure to run on")

    return "C:\\Users\\school\\Desktop\\cross_id\\sdss_mosaic_construction\\SpArcFiRe_output\\table{}\\G.out\\galaxy.csv".format(pa_key)

In [8]:
def _ensure_path_exists(path_to_output,make_ouput_folder_if_not_exists=True):
    if not make_ouput_folder_if_not_exists:
        raise ValueError("The path output is not found {} - make sure you update path_to_output".format(path_to_output))
    
    if not os.path.exists(path_to_output):
        os.makedirs(path_to_output)

In [ ]:
def run_gofher_on_catalog(figure_to_run_on,run_type="",bulge_disk_f=1.0):
    paper_labels = get_paper_dark_side_labels(figure_to_run_on)

    if run_type != "":
        sparcfire_gals = read_sparcfire_galaxy_csv(get_sparcfire_path(figure_to_run_on))

    verbose_header = []
    verbose_rows = []

    ebm_header = []
    ebm_rows = []

    params_header = []
    params_rows = []

    i = 1

    path_to_output = get_path_to_output(figure_to_run_on,run_type=run_type,bulge_disk_f=bulge_disk_f)
    _ensure_path_exists(path_to_output)

    def get_fits_path_cat(name,band):
        """the file path of where existing fits files can be found"""
        return get_fits_path(name,band,figure_to_run_on)

    for name in get_galaxies(figure_to_run_on):

        if standardize_galaxy_name(name) not in paper_labels:
            print("skipping",name)
            continue

        print(name, i,"of",len(get_galaxies(figure_to_run_on)))

        try:
            paper_label = paper_labels[standardize_galaxy_name(name)]
            
            if run_type == "":
                gal = run_gofher(name,get_fits_path_cat,BANDS_IN_ORDER,REF_BANDS_IN_ORDER, paper_label,s=bin_size,bin_prior_to_param_fitting=bin_prior_to_param_fitting)
            else:
                ref_band, inital_gofher_params = get_ref_band_and_gofher_params(sparcfire_gals[name],REF_BANDS_IN_ORDER,bulge_disk_f)
                gal = run_gofher_with_parameters(name,get_fits_path_cat,BANDS_IN_ORDER,ref_band,inital_gofher_params,paper_label=paper_label,mode=run_type)


            if generate_verbose_csv:
                (header,row) = gal.get_verbose_csv_header_and_row(BANDS_IN_ORDER,paper_label)
                if len(verbose_header) == 0: verbose_header = header
                verbose_rows.append(row)

            if generate_ebm_csv:
                (header,row) = gal.get_ebm_csv_header_and_row(BANDS_IN_ORDER, paper_label)
                if len(ebm_header) == 0: ebm_header = header
                ebm_rows.append(row)

            if generate_params_csv:
                (header,row) = gal.get_params_csv_header_and_row()
                if len(params_header) == 0: params_header = header
                params_rows.append(row)

            if generate_visualization:
                save_path = ''
                
                if save_visualization:
                    sub_folder = get_visulization_save_path_folder(figure_to_run_on,run_type,bulge_disk_f)
                    check_if_folder_exists_and_create(sub_folder)
                    save_path = os.path.join(sub_folder,"{}.png".format(name))

                color_image = color = mpimg.imread(get_color_image_path(name,figure_to_run_on))
                visual_string = "type={} f={}".format(run_type,bulge_disk_f)
                visualize(gal,color_image,BANDS_IN_ORDER,paper_label,save_path=save_path,color_flip=(survery_to_use=="sdss"),show_stats=False,visual_string=visual_string)
        except Exception as e:
            print(e)
        i += 1
        
    if generate_verbose_csv:
        verbose_csv_path = os.path.join(path_to_output,"{}_verbose.csv".format(figure_to_run_on))
        write_csv(verbose_csv_path,verbose_header,verbose_rows)

    if generate_ebm_csv:
        ebm_csv_path = os.path.join(path_to_output,"{}_ebm.csv".format(figure_to_run_on))
        write_csv(ebm_csv_path,ebm_header,ebm_rows)

    if generate_params_csv:
        params_csv_path = os.path.join(path_to_output,"{}_params.csv".format(figure_to_run_on))
        write_csv(params_csv_path,params_header,params_rows)



In [ ]:
if not os.path.exists(path_to_catalog_data):
    raise ValueError("The path to the catalog is not found {} - make sure you update path_to_catalog_data".format(path_to_catalog_data))

for figure_to_run_on in figure_to_run_on:
    print("figure",figure_to_run_on)
    for run_type in types_of_runs:
        print("run type",run_type)
        for bulge_disk_f in bulge_disk_fs:
            run_gofher_on_catalog(figure_to_run_on,run_type=run_type,bulge_disk_f=bulge_disk_f)

figure figure8
run type fixed-center
IC1683 1 of 114
IC1755 2 of 114
IC2101 3 of 114
IC5376 4 of 114
MCG-02-02-030 5 of 114
'MCG-02-02-030'
MCG-02-51-004 6 of 114
'MCG-02-51-004'
NGC1035 7 of 114
NGC1056 8 of 114
NGC1084 9 of 114
NGC1093 10 of 114
NGC157 11 of 114
NGC1667 12 of 114
NGC169 13 of 114
NGC2347 14 of 114
Error fitting sersic, using inital_gofher_parameters from sep
NGC2403 15 of 114
'NoneType' object is not subscriptable
NGC2410 16 of 114
NGC2639 17 of 114
NGC2683 18 of 114


c:\Users\school\Desktop\github\gofher\examples\../gofher\ebm.py:63: RuntimeWarning: divide by zero encountered in log
  x = 2.0*sum([-np.log(p) for p in p_values])


NGC2742 19 of 114
NGC2775 20 of 114
NGC2782 21 of 114
'NGC2782'
NGC2841 22 of 114
NGC2903 23 of 114
NGC3160 24 of 114
NGC3198 25 of 114
NGC3227 26 of 114
NGC3310 27 of 114
NGC3368 28 of 114


c:\Users\school\Desktop\github\gofher\examples\../gofher\ebm.py:63: RuntimeWarning: divide by zero encountered in log
  x = 2.0*sum([-np.log(p) for p in p_values])


NGC3521 29 of 114


...

NGC4736 109 of 268
NGC477 110 of 268
NGC4814 111 of 268
NGC488 112 of 268
zero-size array to reduction operation minimum which has no identity
NGC4904 113 of 268
NGC496 114 of 268
NGC4961 115 of 268
NGC497 116 of 268
NGC5000 117 of 268
NGC5016 118 of 268
NGC5056 119 of 268
NGC5204 120 of 268
NGC5205 121 of 268
NGC5218 122 of 268
NGC5371 123 of 268
NGC5378 124 of 268
NGC5383 125 of 268
NGC5394 126 of 268
NGC5406 127 of 268
NGC5457 128 of 268
NGC5474 129 of 268
NGC5480 130 of 268
NGC551 131 of 268
NGC5520 132 of 268
NGC5630 133 of 268
NGC5633 134 of 268
NGC5657 135 of 268
NGC5668 136 of 268
NGC5669 137 of 268
NGC5682 138 of 268
NGC5720 139 of 268
NGC5732 140 of 268
NGC5789 141 of 268
NGC5850 142 of 268
NGC5888 143 of 268
NGC5905 144 of 268
NGC5930 145 of 268
NGC5934 146 of 268
NGC5947 147 of 268
NGC5964 148 of 268
NGC5971 149 of 268
NGC6004 150 of 268
NGC6032 151 of 268
NGC6060 152 of 268
NGC6063 153 of 268
NGC6154 154 of 268
NGC6155 155 of 268
NGC628 156 of 268
NGC6301 157 of 268
NGC647

c:\Users\school\Desktop\github\gofher\examples\../gofher\ebm.py:63: RuntimeWarning: divide by zero encountered in log
  x = 2.0*sum([-np.log(p) for p in p_values])


NGC895 179 of 268
PGC02162 180 of 268
PGC03512 181 of 268
PGC05673 182 of 268
PGC06855 183 of 268
PGC07826 184 of 268
PGC15531 185 of 268
PGC20938 186 of 268
PGC23333 187 of 268
PGC23598 188 of 268
PGC23913 189 of 268
'PGC23913'
PGC24788 190 of 268
PGC26140 191 of 268
PGC26517 192 of 268
PGC27792 193 of 268
PGC28310 194 of 268
PGC28401 195 of 268
PGC31159 196 of 268
PGC32638 197 of 268
PGC33465 198 of 268
PGC36925 199 of 268
PGC38268 200 of 268
PGC38908 201 of 268
PGC39728 202 of 268
'PGC39728'
PGC46767 203 of 268
'PGC46767'
PGC49906 204 of 268
'PGC49906'
PGC55750 205 of 268
'PGC55750'
PGC56010 206 of 268
PGC57931 207 of 268
PGC58410 208 of 268
PGC71106 209 of 268
PGC72144 210 of 268
PGC72453 211 of 268
UGC10310 212 of 268
UGC10796 213 of 268
Error fitting sersic, using inital_gofher_parameters from sep


c:\Users\school\Desktop\github\gofher\gofher_env\Lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\school\Desktop\github\gofher\gofher_env\Lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\school\Desktop\github\gofher\gofher_env\Lib\site-packages\numpy\core\_methods.py:269: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\school\Desktop\github\gofher\gofher_env\Lib\site-packages\numpy\core\_methods.py:226: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\school\Desktop\github\gofher\gofher_env\Lib\site-packages\numpy\core\_methods.py:261: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


UGC1081 214 of 268
UGC10811 215 of 268
UGC1087 216 of 268
UGC11262 217 of 268
UGC12224 218 of 268
UGC12274 219 of 268
UGC12308 220 of 268
UGC12391 221 of 268
UGC12494 222 of 268
UGC12816 223 of 268
UGC12864 224 of 268
UGC1529 225 of 268
UGC1635 226 of 268
UGC1862 227 of 268
UGC3091 228 of 268
UGC312 229 of 268
UGC3140 230 of 268
UGC3253 231 of 268
UGC3973 232 of 268
UGC3995 233 of 268
UGC3997 234 of 268
UGC4107 235 of 268
UGC4256 236 of 268
UGC4284 237 of 268
UGC4368 238 of 268
UGC4380 239 of 268
UGC4458 240 of 268
UGC448 241 of 268
UGC4499 242 of 268
UGC4555 243 of 268
UGC4622 244 of 268
UGC463 245 of 268
UGC4659 246 of 268
'UGC4659'
UGC4936 247 of 268
'UGC4936'
UGC5 248 of 268
UGC5358 249 of 268
UGC5786 250 of 268
UGC6537 251 of 268
UGC6628 252 of 268
UGC6903 253 of 268
UGC6918 254 of 268
UGC7012 255 of 268
UGC7154 256 of 268
